In [1]:
# import os
# os.environ['PYSPARK_PYTHON'] = r"C:\Users\Kiran\AppData\Local\Programs\Python\Python310\python.exe"
# os.environ['PYSPARK_DRIVER_PYTHON'] = r"C:\Users\Kiran\AppData\Local\Programs\Python\Python310\python.exe"
# # Make sure to replace "/path/to/python3.10" with the actual path to your Python executable.

from pyspark.sql import SparkSession

spark = SparkSession.builder.config("spark.driver.host", "localhost").appName("TestApp").getOrCreate()
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df.printSchema()
df.show()


root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [2]:
# Create DataFrame
data = [('James','','Smith','1991-04-01','M',3000),
  ('Michael','Rose','','2000-05-19','M',4000),
  ('Robert','','Williams','1978-09-05','M',4000),
  ('Maria','Anne','Jones','1967-12-01','F',4000),
  ('Jen','Mary','Brown','1980-02-17','F',-1)
]

columns = ["firstname","middlename","lastname","dob","gender","salary"]
df = spark.createDataFrame(data=data, schema = columns)

In [3]:
df.show(3)
# print(df.take(1))

+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|       dob|gender|salary|
+---------+----------+--------+----------+------+------+
|    James|          |   Smith|1991-04-01|     M|  3000|
|  Michael|      Rose|        |2000-05-19|     M|  4000|
|   Robert|          |Williams|1978-09-05|     M|  4000|
+---------+----------+--------+----------+------+------+
only showing top 3 rows


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Test").getOrCreate()
df = spark.range(5).toDF("num")
df.show()

+---+
|num|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [5]:
# Create DataFrame
data = [('James','','Smith','1991-04-01','M',3000),
  ('Michael','Rose','','2000-05-19','M',4000),
  ('Robert','','Williams','1978-09-05','M',4000),
  ('Maria','Anne','Jones','1967-12-01','F',4000),
  ('Jen','Mary','Brown','1980-02-17','F',-1)
]

columns = ["firstname","middlename","lastname","dob","gender","salary"]
df = spark.createDataFrame(data=data, schema = columns)
df.show()

+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|       dob|gender|salary|
+---------+----------+--------+----------+------+------+
|    James|          |   Smith|1991-04-01|     M|  3000|
|  Michael|      Rose|        |2000-05-19|     M|  4000|
|   Robert|          |Williams|1978-09-05|     M|  4000|
|    Maria|      Anne|   Jones|1967-12-01|     F|  4000|
|      Jen|      Mary|   Brown|1980-02-17|     F|    -1|
+---------+----------+--------+----------+------+------+



# Lets start

In [6]:
# !pip install scikit-learn

In [7]:
import logging
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, VectorAssembler, Imputer)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType
from sklearn.metrics import cohen_kappa_score, classification_report
import os
from datetime import datetime



# Create necessary directories if they don't exist
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/kidney_prediction.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('ChronicKidneyDiseasePrediction')

In [8]:
# spark = SparkSession.builder \
#                 .appName("ChronicKidneyDiseasePrediction") \
#                 .config("spark.executor.memory", "4g") \
#                 .config("spark.driver.memory", "4g") \
#                 .getOrCreate()

spark = SparkSession.builder.appName("ChronicKidneyDiseasePrediction").getOrCreate()
logger.info("Spark session initialized successfully")

2025-07-05 12:50:33,607 - ChronicKidneyDiseasePrediction - INFO - Spark session initialized successfully


In [9]:
data_path = r"data/chronic_kidney_disease.csv"

In [10]:
def load_data(file_path):
    """Load data from CSV file"""
    try:
        logger.info(f"Loading data from {file_path}")
        df = spark.read.csv(
            file_path, 
            header=True, 
            inferSchema=True
        )

        # Log basic info about the dataset
        logger.info(f"Data loaded successfully. Shape: ({df.count()}, {len(df.columns)})")
        logger.info(f"Columns: {df.columns}")
        df.printSchema()

        return df
    except Exception as e:
        logger.error(f"Error loading data: {str(e)}")
        raise

# Step 1: Load data
df = load_data(data_path)

2025-07-05 12:50:33,649 - ChronicKidneyDiseasePrediction - INFO - Loading data from data/chronic_kidney_disease.csv
2025-07-05 12:50:35,729 - ChronicKidneyDiseasePrediction - INFO - Data loaded successfully. Shape: (400, 26)
2025-07-05 12:50:35,729 - ChronicKidneyDiseasePrediction - INFO - Columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- bp: double (nullable = true)
 |-- sg: double (nullable = true)
 |-- al: double (nullable = true)
 |-- su: double (nullable = true)
 |-- rbc: string (nullable = true)
 |-- pc: string (nullable = true)
 |-- pcc: string (nullable = true)
 |-- ba: string (nullable = true)
 |-- bgr: double (nullable = true)
 |-- bu: double (nullable = true)
 |-- sc: double (nullable = true)
 |-- sod: double (nullable = true)
 |-- pot: double (nullable = true)
 |-- hemo: double (nullable = true)
 |-- pcv: string (nullable = true)
 |-- wc: string (nullable = true)
 |-- rc: string (nullable = true)
 |-- htn: string (nullable = true)
 |-- dm: string (nullable = true)
 |-- cad: string (nullable = true)
 |-- appet: string (nullable = true)
 |-- pe: string (nullable = true)
 |-- ane: string (nullable = true)
 |-- classification: string (nullable = true)



In [11]:
def preprocess_data(df):
    """Preprocess the raw data"""
    try:
        logger.info("Starting data preprocessing")

        # Convert string columns to numeric where appropriate
        for column in df.columns:
            if df.schema[column].dataType == "string":
                # Check if column contains numeric values stored as strings
                if column in ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 
                             'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']:
                    df = df.withColumn(column, col(column).cast(DoubleType()))

        # Convert target variable to binary (0/1)
        df = df.withColumn("classification", when(col("classification") == "ckd", 1).otherwise(0))

        # Identify numeric and categorical columns
        numeric_cols = [col_name for col_name, dtype in df.dtypes 
                       if dtype in ['int', 'double'] and col_name != "classification"]
        categorical_cols = [col_name for col_name, dtype in df.dtypes 
                          if dtype == 'string']

        logger.info(f"Numeric columns: {numeric_cols}")
        logger.info(f"Categorical columns: {categorical_cols}")

        # Create imputers for missing values
        numeric_imputer = Imputer(
            inputCols=numeric_cols,
            outputCols=[f"{col}_imputed" for col in numeric_cols],
            strategy="mean"
        )

        # String indexers for categorical columns
        indexers = [
            StringIndexer(
                inputCol=column,
                outputCol=f"{column}_indexed",
                handleInvalid="keep"
            ) for column in categorical_cols
        ]

        # Assemble all features
        all_features = [f"{col}_imputed" for col in numeric_cols] + [f"{col}_indexed" for col in categorical_cols]

        assembler = VectorAssembler(
            inputCols=all_features,
            outputCol="features"
        )

        # Create preprocessing pipeline
        preprocessing_pipeline = Pipeline(stages=[numeric_imputer] + indexers + [assembler])

        logger.info("Fitting preprocessing pipeline")
        preprocessor_model = preprocessing_pipeline.fit(df)
        processed_df = preprocessor_model.transform(df)

        # Select only needed columns
        processed_df = processed_df.select("features", "classification")

        logger.info("Data preprocessing completed successfully")
        return processed_df, preprocessor_model

    except Exception as e:
        logger.error(f"Error during data preprocessing: {str(e)}")
        raise


# Step 2: Preprocess data
processed_df, preprocessor = preprocess_data(df)

2025-07-05 12:50:35,760 - ChronicKidneyDiseasePrediction - INFO - Starting data preprocessing
2025-07-05 12:50:35,872 - ChronicKidneyDiseasePrediction - INFO - Numeric columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo']
2025-07-05 12:50:35,874 - ChronicKidneyDiseasePrediction - INFO - Categorical columns: ['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
2025-07-05 12:50:36,040 - ChronicKidneyDiseasePrediction - INFO - Fitting preprocessing pipeline
2025-07-05 12:50:43,563 - ChronicKidneyDiseasePrediction - INFO - Data preprocessing completed successfully


In [12]:
# Step 3: Split data
train_df, test_df = processed_df.randomSplit([0.8, 0.2], seed=42)
logger.info(f"Train size: {train_df.count()}, Test size: {test_df.count()}")

2025-07-05 12:50:45,331 - ChronicKidneyDiseasePrediction - INFO - Train size: 342, Test size: 58


In [13]:
def _calculate_metrics(predictions):
    """Calculate and log comprehensive evaluation metrics"""
    try:
        # Convert to Pandas DataFrame for sklearn metrics
        pred_pandas = predictions.select("prediction", "classification").toPandas()
        y_true = pred_pandas["classification"]
        y_pred = pred_pandas["prediction"]

        # Calculate Cohen's Kappa
        kappa = cohen_kappa_score(y_true, y_pred)
        logger.info(f"Cohen's Kappa Score: {kappa:.4f}")

        # Generate classification report
        report = classification_report(y_true, y_pred, target_names=["No CKD", "CKD"])
        logger.info("Classification Report:\n" + report)

        return kappa, report

    except Exception as e:
        logger.error(f"Error calculating metrics: {str(e)}")
        raise


In [14]:
def train_model(train_df, test_df=None):
    """Train the Random Forest model with cross-validation"""
    try:
        logger.info("Starting model training")

        # Initialize classifier
        rf = RandomForestClassifier(
            labelCol="classification", 
            featuresCol="features", 
            maxDepth=10,
            numTrees=100, 
            maxBins=100, 
            seed=42
        )

        # Create parameter grid
        param_grid = ParamGridBuilder() \
            .addGrid(rf.numTrees, [50, 100, 150]) \
            .addGrid(rf.maxDepth, [5, 10, 15]) \
            .build()

        # Create evaluator
        evaluator_auc = BinaryClassificationEvaluator(
            labelCol="classification",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        )
        
        evaluator_acc = MulticlassClassificationEvaluator(
            labelCol="classification",
            predictionCol="prediction",
            metricName="accuracy"
        )
        
        evaluator_f1 = MulticlassClassificationEvaluator(
            labelCol="classification",
            predictionCol="prediction",
            metricName="f1"
        )
        
        # Create cross-validator
        cv = CrossValidator(
            estimator=rf,
            estimatorParamMaps=param_grid,
            evaluator=evaluator_auc,
            numFolds=5,
            seed=42
        )

        logger.info("Running cross-validation")
        cv_model = cv.fit(train_df)

        # Get best model
        best_model = cv_model.bestModel
        logger.info(f"Best model parameters: {best_model.extractParamMap()}")

        # Evaluate on training data
        train_predictions = best_model.transform(train_df)
        
        # Step 5: Evaluate on train - test set
        # Calculate metrics
        train_auc = evaluator_auc.evaluate(train_predictions)
        train_acc = evaluator_acc.evaluate(train_predictions)
        train_f1 = evaluator_f1.evaluate(train_predictions)

        logger.info(f"Training Metrics - AUC: {train_auc:.4f}, Accuracy: {train_acc:.4f}, F1: {train_f1:.4f}")
        
        # Calculate additional metrics
        train_kappa, train_report = _calculate_metrics(train_predictions)
        
        # If test data is provided, evaluate on test set
        test_metrics = {}
        if test_df:
            test_predictions = best_model.transform(test_df)

            test_auc = evaluator_auc.evaluate(test_predictions)
            test_acc = evaluator_acc.evaluate(test_predictions)
            test_f1 = evaluator_f1.evaluate(test_predictions)

            logger.info(f"Test Metrics - AUC: {test_auc:.4f}, Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")

            test_kappa, test_report = _calculate_metrics(test_predictions)

            test_metrics = {
                'auc': test_auc,
                'accuracy': test_acc,
                'f1': test_f1,
                'kappa': test_kappa,
                'classification_report': test_report
            }

        logger.info("Model training completed successfully")
        # Return model and all metrics 
        # return best_model
        return {
            'model': best_model,
            'train_metrics': {
                'auc': train_auc,
                'accuracy': train_acc,
                'f1': train_f1,
                'kappa': train_kappa,
                'classification_report': train_report
            },
            'test_metrics': test_metrics
        }
        
    except Exception as e:
        logger.error(f"Error during model training: {str(e)}")
        raise

# Step 4: Train model
model_dict = train_model(train_df, test_df=test_df)

2025-07-05 12:50:45,398 - ChronicKidneyDiseasePrediction - INFO - Starting model training
2025-07-05 12:50:45,705 - ChronicKidneyDiseasePrediction - INFO - Running cross-validation
2025-07-05 12:52:18,764 - py4j.clientserver - INFO - Closing down clientserver connection
2025-07-05 12:52:18,780 - ChronicKidneyDiseasePrediction - INFO - Best model parameters: {Param(parent='RandomForestClassifier_76a34bdc7b22', name='bootstrap', doc='Whether bootstrap samples are used when building trees.'): True, Param(parent='RandomForestClassifier_76a34bdc7b22', name='cacheNodeIds', doc='If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval.'): False, Param(parent='RandomForestClassifier_76a34bdc7b22', name='checkpointInterval', doc='set checkpoint interval (>= 1) o

In [15]:
def save_model(model, preprocessor, model_dir="models"):
    """Save the trained model and preprocessor"""
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_path = f"{model_dir}/kidney_model_{timestamp}"

        # Save the complete pipeline (preprocessor + model)
        pipeline = Pipeline(stages=[preprocessor, model])
        pipeline.write().overwrite().save(model_path)

        logger.info(f"Model saved successfully at {model_path}")
        return model_path
    except Exception as e:
        logger.error(f"Error saving model: {str(e)}")
        raise

# Step 6: Save model
model_path = save_model(model_dict["model"], preprocessor)

2025-07-05 12:52:30,663 - ChronicKidneyDiseasePrediction - INFO - Model saved successfully at models/kidney_model_20250705_125221


In [30]:
# model_path, preprocessor, test_auc
model_dict['model'], preprocessor, model_dict['test_metrics']['accuracy'], str(round(model_dict['test_metrics']['accuracy']*100, 4)) + ' %'

(RandomForestClassificationModel: uid=RandomForestClassifier_76a34bdc7b22, numTrees=50, numClasses=2, numFeatures=25,
 PipelineModel_f84484c002cd,
 0.9827586206896551,
 '98.2759 %')

# Deployment

In [34]:
import streamlit as st
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
import os
import logging


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/kidney_app.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('KidneyDiseaseApp')


In [35]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("KidneyDiseasePredictionApp") \
    .getOrCreate()

# Load the trained model
model = PipelineModel.load(model_path)

logger.info("App initialized successfully")

2025-07-05 13:17:18,328 - KidneyDiseaseApp - INFO - App initialized successfully


In [36]:
# Find the latest model
model_dir = "models"
model_files = [f for f in os.listdir(model_dir) if f.startswith("kidney_model")]
print(model_files)

if not model_files:
    st.error("No trained model found. Please train the model first.")
    exit()

['kidney_model_20250705_041332', 'kidney_model_20250705_125221']


In [37]:
# Get the most recent model
latest_model = sorted(model_files)[-1]
print(latest_model)
model_path = os.path.join(model_dir, latest_model)
print(model_path)

kidney_model_20250705_125221
models\kidney_model_20250705_125221


In [44]:
def load_model(model_path):
    """Load the trained pipeline model"""
    logger.info(f"Loading model from {model_path}")
    pipeline_model = PipelineModel.load(model_path)
    model_loaded = True
    logger.info("Model loaded successfully")
    return pipeline_model, model_loaded

logger.info(f"Loading model from {model_path}")
# Load model (update path as needed)
model_path = "models/kidney_model_20250705_125221"  # Update with your model path
pipeline_model, model_loaded = load_model(model_path)

age = st.number_input("Age", min_value=1, max_value=120, value=50)
bp = st.number_input("Blood Pressure (mm Hg)", min_value=50, max_value=250, value=80)
sg = st.selectbox("Specific Gravity", [1.005, 1.010, 1.015, 1.020, 1.025])
al = st.selectbox("Albumin (0-5)", [0, 1, 2, 3, 4, 5])
su = st.selectbox("Sugar (0-5)", [0, 1, 2, 3, 4, 5])
rbc = st.selectbox("Red Blood Cells", ["normal", "abnormal"])
pc = st.selectbox("Pus Cells", ["normal", "abnormal"])
pcc = st.selectbox("Pus Cell Clumps", ["present", "notpresent"])
ba = st.selectbox("Bacteria", ["present", "notpresent"])
bgr = st.number_input("Blood Glucose Random (mg/dL)", min_value=50, max_value=500, value=100)
bu = st.number_input("Blood Urea (mg/dL)", min_value=10, max_value=300, value=40)
sc = st.number_input("Serum Creatinine (mg/dL)", min_value=0.5, max_value=20.0, value=1.0, step=0.1)
sod = st.number_input("Sodium (mEq/L)", min_value=100, max_value=200, value=140)
pot = st.number_input("Potassium (mEq/L)", min_value=2.0, max_value=10.0, value=4.0, step=0.1)
hemo = st.number_input("Hemoglobin (g/dL)", min_value=3.0, max_value=20.0, value=12.0, step=0.1)
pcv = st.number_input("Packed Cell Volume", min_value=10, max_value=60, value=40)
wc = st.number_input("White Blood Cell Count (cells/cumm)", min_value=2000, max_value=20000, value=8000)
rc = st.number_input("Red Blood Cell Count (millions/cmm)", min_value=2.0, max_value=8.0, value=4.5, step=0.1)
htn = st.selectbox("Hypertension", ["yes", "no"])
dm = st.selectbox("Diabetes Mellitus", ["yes", "no"])
cad = st.selectbox("Coronary Artery Disease", ["yes", "no"])
appet = st.selectbox("Appetite", ["good", "poor"])
pe = st.selectbox("Pedal Edema", ["yes", "no"])
ane = st.selectbox("Anemia", ["yes", "no"])


input_data = {
                    "age": float(age),
                    "bp": float(bp),
                    "sg": float(sg),
                    "al": float(al),
                    "su": float(su),
                    "rbc": rbc,
                    "pc": pc,
                    "pcc": pcc,
                    "ba": ba,
                    "bgr": float(bgr),
                    "bu": float(bu),
                    "sc": float(sc),
                    "sod": float(sod),
                    "pot": float(pot),
                    "hemo": float(hemo),
                    "pcv": float(pcv),
                    "wbcc": float(wbcc),
                    "rbcc": float(rbcc),
                    "htn": htn,
                    "dm": dm,
                    "cad": cad,
                    "appet": appet,
                    "pe": pe,
                    "ane": ane
                }


2025-07-05 19:31:08,158 - KidneyDiseaseApp - INFO - Loading model from models/kidney_model_20250705_125221
2025-07-05 19:31:08,161 - KidneyDiseaseApp - INFO - Loading model from models/kidney_model_20250705_125221
2025-07-05 19:31:13,372 - KidneyDiseaseApp - INFO - Model loaded successfully


In [47]:
def preprocess_input(input_data):
    """Preprocess user input using the pipeline"""
    try:
        # Convert to Spark DataFrame
        input_df = spark.createDataFrame(input_data)

        # Apply the full pipeline (preprocessing + model)
        #processed_df = pipeline_model.transform(input_df)

        return input_df # processed_df
    except Exception as e:
        logger.error(f"Error preprocessing input: {str(e)}", exc_info=True)
        raise

def predict(input_data):
    """Make prediction on input data"""
    try:
        if not model_loaded:
            raise ValueError("Model not loaded")

        # Preprocess and predict
        processed_df = preprocess_input(input_data)
        prediction = processed_df.collect()[0]

        return {
            'prediction': int(prediction['prediction']),
            'probability': float(prediction['probability'][1]),  # Probability of CKD
            'raw_prediction': [float(x) for x in prediction['rawPrediction']]
        }
    except Exception as e:
        logger.error(f"Prediction failed: {str(e)}", exc_info=True)
        raise
        
# Make prediction
result = predict([input_data])  # Wrap in list for single prediction
result

2025-07-05 20:04:09,134 - KidneyDiseaseApp - ERROR - Prediction failed: prediction
Traceback (most recent call last):
  File "C:\ProjectWork\Basic_Python\Neha Project\Chronic Kidney Disease Prediction Project using PySpark\venv\lib\site-packages\pyspark\sql\types.py", line 3124, in __getitem__
    idx = self.__fields__.index(item)
ValueError: 'prediction' is not in list

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Kiran\AppData\Local\Temp\ipykernel_12820\2521976259.py", line 26, in predict
    'prediction': int(prediction['prediction']),
  File "C:\ProjectWork\Basic_Python\Neha Project\Chronic Kidney Disease Prediction Project using PySpark\venv\lib\site-packages\pyspark\sql\types.py", line 3129, in __getitem__
    raise PySparkValueError(item)
pyspark.errors.exceptions.base.PySparkValueError: prediction


PySparkValueError: prediction

# *Need to work on preprocessing